In [2]:
import pandas as pd
import ast

In [3]:
df = pd.read_csv("./data/materials_data.csv")

# Rare earth elements
rare_earths = ["Nd","Sm","Gd","Eu","Tb","Dy","Ho","Er","Pr","La","Ce","Y","Sc"]

len(df)

41838

In [4]:
DENSITY_MAX = 7.5      # g/cm^3, upper bound (lighter than or ~ NdFeB)
DENSITY_MIN = 0.5       # sanity floor to exclude bad/near-empty structures
ENERGY_ABOVE_HULL_MAX = 0.05  # eV/atom, stability filter (near convex hull)

In [5]:
#Filter candidates by density and energy above hull
df_filtered = df[(df["density"] <= DENSITY_MAX) &
                 (df["density"] >= DENSITY_MIN) &
                 (df["energy_above_hull"] <= ENERGY_ABOVE_HULL_MAX)]
len(df_filtered)

12186

In [6]:
# Average atomic weight per atom, derived directly from already-downloaded fields
df_filtered["avg_atomic_weight"] = df_filtered["density"] / df_filtered["density_atomic"]

# Total mass of the cell (average atomic weight * number of atoms in cell)
df_filtered["cell_mass"] = df_filtered["avg_atomic_weight"] * df_filtered["nsites"]

# True specific magnetic performance index: moment per unit mass
df_filtered["magnetic_performance_index"] = (
    df_filtered["total_magnetization"] / df_filtered["cell_mass"]
)

df_filtered = df_filtered.sort_values("magnetic_performance_index", ascending=False)
len(df_filtered)

12186

In [7]:
df_filtered = df_filtered.copy()

def normalize_elements(elements):
    if isinstance(elements, str):
        try:
            return ast.literal_eval(elements)
        except (SyntaxError, ValueError):
            return [part.replace("Element ", "").strip() for part in elements.strip("[]").split(",")]
    return [str(el).replace("Element ", "").strip() for el in elements]

df_filtered["elements"] = df_filtered["elements"].apply(normalize_elements)
df_filtered["elements"].head(10)

8541         [Cl, Fe]
10240      [K, Mn, S]
34600    [Mn, Rb, Te]
34599     [Mn, Rb, S]
28705    [Cd, Mn, Te]
6214      [Cs, Mn, S]
34022             [O]
8542         [Cl, Fe]
34023             [O]
28304     [Cr, I, Mn]
Name: elements, dtype: object

In [8]:
df_filtered['formula'].head(10)

8541         FeCl3
10240      K2Mn3S4
34600    Rb2Mn3Te4
34599     Rb2Mn3S4
28705     Mn3CdTe4
6214      Cs2Mn3S4
34022           O2
8542         FeCl3
34023           O2
28304      Mn2CrI6
Name: formula, dtype: str

- FeCl3, K2Mn3S4, Rb2Mn3Te4, Cs2Mn3S4 — these are ionic salts. They're magnetic in the DFT sense, but chemically they're often hygroscopic, chemically unstable, and not something you'd ever put into a motor or device. High mass-specific magnetization here just reflects that Cl/S/Te are light-ish and the compound has some unpaired spins — it says nothing about whether this is a usable magnet.
- O2 (mp-2206907, mp-2739215) — solid oxygen is only magnetically ordered as a cryogenic solid phase (below ~54K). This is a real red flag: it means your filter is picking up materials that are magnetic only in exotic/low-temperature phases, completely irrelevant for a room-temperature permanent magnet application.
- Mn2CrI6 — a van der Waals magnetic material (iodides), interesting for 2D magnetism research but not a bulk permanent magnet candidate either.

In [13]:
EXCLUDE_ELEMENTS_PRACTICAL = {
    "Li", "Na", "K", "Rb", "Cs",
    "Cl", "Br", "I", "F",
    "S", "Se", "Te",
}

def is_practical_composition(elements_list):
    symbols = [str(el) for el in elements_list]   # force plain string symbols
    return not any(sym in EXCLUDE_ELEMENTS_PRACTICAL for sym in symbols)

df_practical = df_filtered[df_filtered["elements"].apply(is_practical_composition)]
df_practical = df_practical.sort_values("magnetic_performance_index", ascending=False)

print(f"Remaining after practicality filter: {len(df_practical)}")
print(df_practical['formula'].head(10))

Remaining after practicality filter: 3672
34022           O2
34023           O2
34020           O2
34019           O2
8997         FePO4
2679     Ca(MnGe)2
8987         FePO4
8989         FePO4
724      Ba(FeO2)2
8986         FePO4
Name: formula, dtype: str


A permanent magnet needs at least a transition metal + something else forming a real lattice; pure elemental phases like O2 aren't candidates.

In [14]:
df_practical = df_practical[df_practical["nelements"] >= 2]
df_practical = df_practical.sort_values("magnetic_performance_index", ascending=False)

print(f"Remaining after single-element exclusion: {len(df_practical)}")
print(df_practical['formula'].head(10))

Remaining after single-element exclusion: 3668
8997         FePO4
2679     Ca(MnGe)2
8987         FePO4
8989         FePO4
724      Ba(FeO2)2
8986         FePO4
9004         FePO4
9005         FePO4
34113          PH3
8992         FePO4
Name: formula, dtype: str


PH3 (phosphine) — mp-696588\
This is a molecular gas (phosphine), with no transition metal at all — just P and H. It's magnetic in the DFT sense but chemically irrelevant as a permanent magnet candidate, same category of problem as the O2 entries.

require at least one genuine magnetic transition metal to be present. This is more robust and future-proof:

In [15]:
REQUIRED_MAGNETIC_ELEMENTS = {
    "Fe", "Co", "Ni", "Mn", "Cr",   # primary 3d ferromagnetic/ferrimagnetic elements
}

def has_magnetic_element(elements_list):
    symbols = [str(el) for el in elements_list]
    return any(sym in REQUIRED_MAGNETIC_ELEMENTS for sym in symbols)

df_practical = df_practical[df_practical["elements"].apply(has_magnetic_element)]
df_practical = df_practical.sort_values("magnetic_performance_index", ascending=False)

print(f"Remaining after requiring a magnetic transition metal: {len(df_practical)}")
print(df_practical['formula'].head(10))

Remaining after requiring a magnetic transition metal: 2552
8997        FePO4
2679    Ca(MnGe)2
8987        FePO4
8989        FePO4
724     Ba(FeO2)2
8986        FePO4
9004        FePO4
9005        FePO4
8992        FePO4
8998        FePO4
Name: formula, dtype: str


In [16]:
df_practical.to_csv("./data/lightweight_fm_candidates_practical.csv", index=False)

In [12]:
# See best (highest-scoring) representative of each unique formula
df_unique_formula = df_practical.sort_values("magnetic_performance_index", ascending=False).drop_duplicates(
    subset="formula", keep="first")
print(f"Unique formulas represented: {df_unique_formula['formula'].nunique()}")
print(df_unique_formula.head(15))

Unique formulas represented: 2027
      material_id       formula   density  energy_above_hull  \
8997    mp-753756         FePO4  2.389475           0.040410   
2679     mp-19824     Ca(MnGe)2  5.064021           0.000000   
724      mp-19285     Ba(FeO2)2  4.284987           0.035433   
5278    mp-771134         CoPO4  2.321093           0.049185   
28216   mp-532236   Mn17Fe13O40  4.619103           0.049598   
29061  mp-1222437    Mn5(FeO3)4  4.645185           0.049518   
2170     mp-22427   BaSr(FeO2)4  4.349907           0.023438   
7577   mp-2912507       Fe2C2O7  2.581034           0.049256   
937     mp-652683     Ba2Fe6O11  4.626406           0.009605   
1716     mp-18950   BaCa(FeO2)4  4.245488           0.032707   
28236   mp-757698     Mn23FeO32  4.572721           0.013527   
7956   mp-1181813         Fe3O4  4.797830           0.045893   
1734   mp-1193438     BaCaFe4O7  4.436038           0.027601   
1309   mp-1228526  Ba3Fe10SnO20  4.746417           0.046716   
29280 